# Practical 3 - Planning

In this practical session we will plan in an environment with a rack. Before we start read the following prerequisite:

In [ ]:
!current_env=$(conda info --envs | grep "*" | awk '{print $1}'); if [ "$current_env" = "irm" ]; then echo "The current active Conda environment is 'irm'. You can run the next cell."; else echo "The current active Conda environment is not 'irm'. Make sure to run this notebook with your irm conda env (conda activate irm)"; fi

## Part 1 - Grasping your first object

Power on your robot, make sure your robot is first in **local control mode** and complete the following task:
<div class="alert alert-block alert-info"> <b>Task 1:</b> Configure the robot TCP length and payload. The tool length is 50 mm when closed. The tool's weight can be ignored. When finished, set your robot to remote control mode.</div>

Configured? Now go to **remote control mode**

In [ ]:
import os
import random
import sys
from pathlib import Path
import numpy as np
import math
from loguru import logger

import airo_models
from airo_drake import finish_build
from airo_drake import visualize_frame
from airo_models.urdf import replace_value, make_static
from airo_robots.grippers.hardware.robotiq_2f85_urcap import Robotiq2F85
from airo_robots.manipulators.hardware.ur_rtde import URrtde
from pydrake.geometry import MeshcatVisualizer, MeshcatVisualizerParams, Role
from pydrake.math import RollPitchYaw
from pydrake.multibody.tree import FixedOffsetFrame
from pydrake.planning import RobotDiagramBuilder
from pydrake.geometry import Meshcat
from pydrake.math import RigidTransform, RotationMatrix

# Change the minimum logging level
logger.remove()
logger.add(sys.stdout, format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {message}",
           filter=lambda record: record["level"].name == "INFO")

# Some helper functions. You can ignore these. 

def add_meshcat(robot_diagram_builder: RobotDiagramBuilder, meshcat=None) -> Meshcat:
    scene_graph = robot_diagram_builder.scene_graph()
    builder = robot_diagram_builder.builder()

    # Adding Meshcat must be done before finalizing
    meshcat = Meshcat() if meshcat is None else meshcat
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)

    # Add visualizer for proximity/collision geometry
    collision_params = MeshcatVisualizerParams(role=Role.kProximity, prefix="collision", visible_by_default=False)
    MeshcatVisualizer.AddToBuilder(builder, scene_graph.get_query_output_port(), meshcat, collision_params)

    return meshcat


def _make_static(urdf):
    joint_types = ["revolute", "continuous", "prismatic", "floating", "planar"]
    for joint_type in joint_types:
        replace_value(urdf, "@type", joint_type, "fixed")
    return urdf

In [ ]:
# setup visualization
meshcat = Meshcat()

In [ ]:
ip_address = "10.42.0.162"  # fill in the robot IP
try:
    robot = URrtde(ip_address, URrtde.UR3E_CONFIG)
except RuntimeError as e:
    robot = None
    gripper = None
    raise e

## Part 2 - Setting a robot scene

We will now put together a robot scene with a rack. Therefore we build you a simple rack that can be mounted on your workbench. We will provide screws for this.

<div class="alert alert-block alert-info"> <b>Task 2:</b> Mount your rack on the table (in the real world, we will do simulation later in the practical) in a way that both shelves are EASILY within reach of the UR3e. If necessary, do some tests with the free drive mode.</div>

In your previous homework you defined an URDF file with a rack. The URDF of the rack, the table and the robot form together the robot scene. We need them, so that the planner we will implement can perform collision checks.

<div class="alert alert-block alert-info"> <b>Task 3:</b> Study and modify the code below so that your scene matches the real world. Use your previous homework to import your rack.urdf file. Position it accordingly in the simulator. In addition, configure the robot to be at the appropriate pose. </div>

In [ ]:
#TODO(@Student): change all offsets and rotations depending on your setup

# Table -> Mounting plate of the robot
x_offset = 0.10
y_offset = 0.20

rpy_table_mountingplate = [0, 0, 0]
xyz_table_mountingplate = [x_offset, y_offset, 0.0]

X_Table_MountingPlate = RigidTransform(RollPitchYaw(*rpy_table_mountingplate), xyz_table_mountingplate)

# Table -> Rack 
x_offset = 0.2
y_offset = 0.5
z_offset = 0.5

roll = 0
pitch = 0
yaw = 0

X_Table_MountingRack = RigidTransform(RollPitchYaw(roll, pitch, yaw), [x_offset, y_offset, z_offset])

# Table -> Micro plate
x_offset = 0.1
y_offset = 0.1
z_offset = 0.1

roll = -np.pi/2
pitch = -np.pi/2
yaw = 0
X_Table_UPlate = RigidTransform(RollPitchYaw([roll, pitch, yaw]), [x_offset, y_offset, z_offset])

# Robot starting position
init_q = [0, -np.pi / 2, 0, -np.pi / 2, 0, 0] if robot is None else robot.get_joint_configuration()

In [ ]:
# Init env
robot_diagram_builder = RobotDiagramBuilder()
plant = robot_diagram_builder.plant()
parser = robot_diagram_builder.parser()
parser.SetAutoRenaming(True)

meshcat.Delete()
meshcat.Delete("/scenario")
meshcat.DeleteAddedControls()
add_meshcat(robot_diagram_builder, meshcat)

# 1. Find your models
table_urdf_path = str(Path(os.path.join("resources", "table.urdf")).resolve())
mounting_plate_urdf_path = str(Path(os.path.join("resources", "mounting_plate.urdf")).resolve())
arm_urdf_path = airo_models.get_urdf_path("ur3e")
gripper_urdf_path = airo_models.get_urdf_path("robotiq_2f_85")
gripper_urdf = airo_models.urdf.read_urdf(gripper_urdf_path)
make_static(gripper_urdf)
gripper_urdf_path = airo_models.urdf.write_urdf_to_tempfile(
    gripper_urdf, gripper_urdf_path, prefix="robotiq_2f_85_static_"
)
gripper_urdf_path = str(Path(os.path.join("resources", "pipette.urdf")).resolve())
rack_urdf_path = str(Path(os.path.join("resources", "rack.urdf")).resolve())
uplate_urdf_path = str(Path(os.path.join("resources", "uplate.urdf")).resolve())

# 2. Add models to simulator
table_index = parser.AddModels(table_urdf_path)[0]
mounting_plate_index = parser.AddModels(mounting_plate_urdf_path)[0]
arm_index = parser.AddModels(arm_urdf_path)[0]
gripper_index = parser.AddModels(gripper_urdf_path)[0]
rack_index = parser.AddModels(rack_urdf_path)[0]
uplate_index = parser.AddModels(uplate_urdf_path)[0]

# 3. Get mounting and attach frames for all models
world_frame = plant.world_frame()
table_frame = plant.GetFrameByName("base_link", table_index)
mounting_plate_frame = plant.GetFrameByName("base_link", mounting_plate_index)
mounting_plate_top_centre_frame = plant.GetFrameByName("plate_top_centre",
                                                       mounting_plate_index)  # this is a virtual helper frame 
arm_frame = plant.GetFrameByName("base_link", arm_index)
arm_tool_frame = plant.GetFrameByName("tool0", arm_index)
gripper_frame = plant.GetFrameByName("base_link", gripper_index)
rack_frame = plant.GetFrameByName("base_link", rack_index)
uplate_frame = plant.GetFrameByName("base_link", uplate_index)

# 4. Define relative transform, i.e. specify child-parent transforms
X_MountingPlateTopCentre_ArmBaseLink = RigidTransform(RollPitchYaw(0, 0, -np.pi / 2), [0, 0, 0])
X_Tool0_GripperBase = RigidTransform(RollPitchYaw(0, 0, 0), [0, 0, 0])

# Add TCP frame
tcp_offset = 0.05
# TCP is between the fingers and tcp_offset cm from the wrist towards the gripper forwards direction (Z+ in TCP frame)
X_Tool0Tcp = RigidTransform(RotationMatrix.Identity(), [0, 0, tcp_offset])
arm_tcp_frame = plant.AddFrame(FixedOffsetFrame("tcp", plant.GetFrameByName("tool0", arm_index), X_Tool0Tcp))

# 5. Weld frames together
plant.WeldFrames(world_frame, table_frame)  # world -> table
plant.WeldFrames(table_frame, mounting_plate_frame, X_Table_MountingPlate)  # table -> mountingplate
plant.WeldFrames(mounting_plate_top_centre_frame, arm_frame,
                 X_MountingPlateTopCentre_ArmBaseLink)  # mountingplate -> arm
plant.WeldFrames(arm_tool_frame, gripper_frame, X_Tool0_GripperBase)  # arm -> gripper 
plant.WeldFrames(table_frame, rack_frame, X_Table_MountingRack)  # table -> rack
plant.WeldFrames(table_frame, uplate_frame, X_Table_UPlate)  # table -> uplate

# Finish the build of the simulation env
robot_diagram, context = finish_build(robot_diagram_builder, meshcat)

# robot arm init joint positions
plant_context = plant.GetMyContextFromRoot(context)
plant.SetPositions(plant_context, arm_index, init_q)
robot_diagram.ForcedPublish(context)

Next, we will show you where the origin of the `World` is. In our case, the origin of the `World` is the origin of the `Table`. Hence, when we express a frame relative to the World, we are expressing relative to this point. Keep this in mind. This is not the same as expressing frames relative to the robots base, which is also shown.  

In [ ]:
# lets demonstrate the origin of the world, which is the origin of the table (base_link)
table = plant.GetModelInstanceByName("table")
X_W_T = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(), plant.GetBodyByName("base_link", table))
visualize_frame(meshcat, "/scenario/planner_origin", X_W_T, length=0.35, radius=0.01)
X_W_B = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                  plant.GetBodyByName("base", plant.GetModelInstanceByName("ur3e")))
visualize_frame(meshcat, "/scenario/arm_base", X_W_B, length=0.20, radius=0.005)

We have a utility class that we will define globally for this notebook; `RobotKinematics`. As the name implies, the `RobotKinematics` class provides kinematic functions such as forward kinematics and inverse kinematics based on the simulation environment (called `Plant` in our simulators jargon).

In [ ]:
from kinematics import RobotKinematics

robot_kinematics = RobotKinematics(robot_diagram, arm_index, meshcat)

For example, we can use it to evaluate a forward kinematics call and examine to which joint configuration it corresponds. The result of kinematic queries will be visualized in meshcat.

In [ ]:
robot_kinematics.forward_kinematics([0, 0, 0, 0, 0, 0])
print(
    f"Look at what the forward kinematics like on {robot_kinematics.meshcat.web_url()}. (Do not confuse this with your main simulator and visualization running at {meshcat.web_url()}. These do not intercommunicate their data with each other.)")

<div class="alert alert-block alert-info"> <b>Task 4:</b> Configure your robot to local mode again and use the freedrive function to verify your scene settings. Tip: with the init_q variable in the above code block you can configure your robot so that it matches the real world one. </div>

(you can also use the python commands to put the robot into freedrive so you do not have to restart the kernel of this notebook. See previous practicals.)

Everything correct? Then you finished part 2. In the code below we load some additional libraries and make helper functions to draw lines in the Meshcat. This will be of help in the next parts of this session.

In [ ]:
import numpy as np
from treelib import Node, Tree

from pydrake.geometry import Cylinder
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.planning import SceneGraphCollisionChecker
from pydrake.geometry import Rgba, Meshcat
from typing import List


def draw_lines(meshcat, name: str, from_to_list, thickness: float, color: List):
    """
    :param from_to_list: [ [from,to ], [from, to], ... ] can be extracted from tree by iterating over each node and gettings its parent
    """
    from_to_list = np.array(from_to_list).reshape((-1, 2, 3))

    starts = from_to_list[:, 0, :].reshape(3, -1)
    ends = from_to_list[:, 1, :].reshape(3, -1)

    meshcat.SetLineSegments(name, starts, ends, thickness, Rgba(*color))


def draw_line_between_points(meshcat: Meshcat, name: str, start: np.ndarray, end: np.ndarray, thickness: float,
                             color: List):
    vertices = np.array([start, end]).T
    meshcat.SetLine(name, vertices, thickness, Rgba(*color))


def AddMeshcatTriad(meshcat, path, length=0.05, radius=0.002, opacity=1.0, X_PT=RigidTransform()):
    meshcat.SetTransform(path, X_PT)
    # x-axis
    X_TG = RigidTransform(RotationMatrix.MakeYRotation(np.pi / 2), [length / 2.0, 0, 0])
    meshcat.SetTransform(path + "/x-axis", X_TG)
    meshcat.SetObject(path + "/x-axis", Cylinder(radius, length), Rgba(1, 0, 0, opacity))

    # y-axis
    X_TG = RigidTransform(RotationMatrix.MakeXRotation(np.pi / 2), [0, length / 2.0, 0])
    meshcat.SetTransform(path + "/y-axis", X_TG)
    meshcat.SetObject(path + "/y-axis", Cylinder(radius, length), Rgba(0, 1, 0, opacity))

    # z-axis
    X_TG = RigidTransform([0, 0, length / 2.0])
    meshcat.SetTransform(path + "/z-axis", X_TG)
    meshcat.SetObject(path + "/z-axis", Cylinder(radius, length), Rgba(0, 0, 1, opacity))


collision_checker = SceneGraphCollisionChecker(
    model=robot_diagram,
    robot_model_instances=[arm_index, gripper_index],
    edge_step_size=0.125,  # Arbitrary value, unused.
    env_collision_padding=0.005,
    self_collision_padding=0.005)


## Part 3 - a basic RRT planner in configuration space

From the course you know that RRT, Rapidly exploring Random Tree, is an algorithm to search high-dimensional spaces (e.g., configuration space, work space, etc) by building a space-filling tree. Assume we will plan in configuration space, then in its most basic form, an RRT tree (T) grows from the starting configuration (q_init) to the goal configuration (q_goal) by following the following steps:

```python
T.add_node(q_init)
for i < max_number_of_iterations:
    if random.uniform(0,1) < p:
        q_next = random_configuration()
    else:
        q_next = q_goal
    q_near = closest_q_on_tree(T, q_next)
    q_new = grow_closer(q_near, q_next, Δq)
    if collision_checks_ok():
        T.add_node(q_new)
        T.add_edge(q_near, q_new)     
    if(q_new is close to q_goal):
        return T
```

In the above pseudocode, Δq, is an incremental distance in configuration space. The algorithm stops when q_new is close to q_goal or when a maximal number of iterations is completed. We will now implement the necessary functions one by one. Note that we added an extra trick: with some probability p, we sample towards the goal instead of a random direction. This balances exploration with exploitation and is called goal biasing.

We start by randomly sampling a configuration for the UR3e. From the previous practicals you know that the joints, except for *wrist 3*, have a range from $-2\pi$ to $2\pi$. 

<div class="alert alert-block alert-info"> <b>Task 5:</b> Return a random configuration and return this as a list q = [q0,q1,...,q5]</div>

In [ ]:
def sample_random_configuration(q_goal):
    return #TODO(@students)

q_random = sample_random_configuration(np.zeros(6))
robot_kinematics.forward_kinematics(q_random)
print(f"Random sampled configuration ({q_random} visible at {robot_kinematics.meshcat.web_url()})")

Another important aspect of RRT is finding a distance metric. For simplicity we use the Euclidean distance. 

<div class="alert alert-block alert-info"> <b>Task 6:</b> Implement the Euclidian distance between two configurations q1 and q2. Why is Euclidian distance not optimal in our case?</div>

Your answer here

In [ ]:
def calc_eucl_dist(q1, q2):
    return #TODO(@students)

<div class="alert alert-block alert-info"> <b>Task 7:</b> Below we provide another distance metric. Choose some interesting configurations q_test_1 and q_test_2 and compare the outcome of both distance metrics. What do you notice?</div>

In [ ]:
def calc_torus_distance(p1, p2):
    L = np.array([2 * np.pi] * 6)
    dp = np.abs(np.array(p2) - np.array(p1))
    dp = np.minimum(dp, L - dp)
    return np.sqrt(np.sum(dp ** 2))

In [ ]:
q_test_1 = np.zeros((6,))
q_test_2 = np.full((6,), np.pi / 2)

print(f"{calc_eucl_dist(q_test_1, q_test_2)}")
print(f"{calc_torus_distance(q_test_1, q_test_2)}")

Your answer here

In [ ]:
# change here if you want to test another distance metric in your RRT algorithm
def calculate_distance(q1, q2):
    #return calc_torus_distance(q1, q2)
    return calc_eucl_dist(q1, q2)

To implement the RRT tree we use a very simple python library [treelib](https://treelib.readthedocs.io/en/latest/). Have a quick look at it's API (click the link) and then complete the following task:

<div class="alert alert-block alert-info"> <b>Task 8:</b> Study the code below and make sure you understand what happens.</div>

In [ ]:
def find_closest_node_id(tree, target_q):
    """
    Finds the closest child node to the target point based on the calculate_distance() function defined before.

    Args:
        tree (Tree): The tree containing the nodes.
        target_point (numpy.ndarray): The target point to find the closest child for.

    Returns:
        Node: The closest child node.
    """
    closest_distance = float('inf')
    closest_child_id = tree.root

    # Recursive function to iterate over all children
    def iterate_all_children(node):
        nonlocal closest_distance, closest_child_id
        for child_node in tree.children(node.identifier):  # 'tree' is accessible here
            distance = calculate_distance(child_node.tag, target_q)
            if distance < closest_distance:
                closest_distance = distance
                closest_child_id = child_node.identifier
            iterate_all_children(child_node)

    iterate_all_children(tree[tree.root])

    return closest_child_id

We now have to find q_new which lies on the line between q_near and q_rand. 

<div class="alert alert-block alert-info"> <b>Task 9:</b> Study the code below and make sure you understand what happens. Is this also valid when the torus-metric is used? + explain your answer</div>

Your answer here

In [ ]:
def find_point_on_line(q1, q2, d):
    """
    Finds a point on the line defined by two points, at a distance 'd' from 'q1'.

    Args:
        q1 (numpy.ndarray): The first configuration (6 angles).
        q2 (numpy.ndarray): The second configuration (6 angles).
        d (float): The distance from 'point1' along the line.

    Returns:
        tuple: The coordinates of the point at distance 'd' from 'q1' along the line.
    """
    # Calculate the direction vector of the line
    direction_vector = q2 - q1

    # Calculate the length of the line segment
    line_length = calculate_distance(q1, q2)

    # Calculate the scaling factor for the direction vector
    scaling_factor = d / line_length

    # Calculate the coordinates of the point at distance 'd' along the line
    q_new = q1 + scaling_factor * direction_vector

    return q_new

<div class="alert alert-block alert-info"> <b>Task 10:</b> Study and complete the code below so that you have a working RRT function.</div>

Note: this function takes the following arguments:

- q_init: the initial joint configuration the robot is in
- q_goal: the goal joint configuration
- delta_q: the step size
- max_iterations: for how many steps do we run RRT?
- p: the probability that we go in the direction of the goal configuration
- tolerance: the convergence criteria

In [ ]:
def plan_with_rrt(q_init, q_goal, delta_q=np.deg2rad(2), max_iterations=5_000, p=0.05, tolerance=np.deg2rad(2)):
    # Init the tree and the MeshCat
    rrt_path = []
    rrt_tree = Tree()
    meshcat.Delete("rrt")
    meshcat.Delete("rrt_path")

    plant_context = plant.GetMyContextFromRoot(context)

    plant.SetPositions(plant_context, arm_index, q_goal)
    robot_diagram.ForcedPublish(context)
    X_W_TCP = plant.CalcRelativeTransform(plant_context, plant.world_frame(), arm_tcp_frame)

    # Make sure tcp_start and tcp_target are in the correct base. They are drawn in the world space but it might be that tcp_start is actually X_B_TCP and not X_W_TCP
    X_W_B = plant.EvalBodyPoseInWorld(plant_context, plant.GetBodyByName("base"))
    X_B_TCP = X_W_B.inverse() @ RigidTransform(X_W_TCP)

    plant.SetPositions(plant_context, arm_index, q_init)
    robot_diagram.ForcedPublish(context)

    q_start = plant.GetPositions(plant_context, arm_index)
    X_W_TcpStart = plant.CalcRelativeTransform(plant_context, plant.world_frame(), arm_tcp_frame)

    # Add the start and goal frames to ease the debugging process
    AddMeshcatTriad(meshcat, "rrt/start", X_PT=X_W_TcpStart)
    AddMeshcatTriad(meshcat, "rrt/end", X_PT=X_W_TCP)

    # Create the root node based
    root_data = q_start
    root_node = Node(tag=root_data, identifier="root")
    rrt_tree.add_node(root_node)
    child_id = root_node.identifier

    last_child = None

    # Loop for a number of iterations
    for i in range(max_iterations):
        print(f"iteration {i}")
        child_id = str(i)

        # TODO(@Student): Complete the missing steps of the RRT algorithm (7 lines exactly):
        #     We want you to name two specific variables to use downstream in our code:
        # (1) parent_id: the id of the parent node you want to add a new child to (see next line of comment)
        # (2) possibly_new_node: the new node you want to add to the tree which we will downstream check for collision
        
        # Start task here:

        
        # End of the Task (make sure you defined variables "parent_id" and "possibly_new_node"! 


        # Check collisions and add the new node
        if collision_checker.CheckConfigCollisionFree(possibly_new_node):
            last_child = rrt_tree.create_node(tag=possibly_new_node, identifier=child_id, parent=parent_id)

            _start = X_W_B @ RigidTransform(robot_kinematics.forward_kinematics(last_child.tag))
            _end = X_W_B @ RigidTransform(
                robot_kinematics.forward_kinematics(rrt_tree.parent(last_child.identifier).tag))
            draw_line_between_points(meshcat, f"rrt/{child_id}", _start.translation(), _end.translation(),
                                     0.01, [1, 1, 1, 1])

            X_B_possible_new_node = RigidTransform(robot_kinematics.forward_kinematics(possibly_new_node))
            X_W_possible_new_node = X_W_B @ X_B_possible_new_node

            task_space_dist = calc_eucl_dist(X_W_possible_new_node.translation(), RigidTransform(X_W_TCP).translation())

            #print(f"Cspace distance to goal of new node: {calculate_distance(q_goal, possibly_new_node)}")
            #print(f"Task space distance to goal of new node: {task_space_dist}")

            # Stop when we reached the goal configuration:
            if calculate_distance(q_goal, possibly_new_node) < tolerance:
                print("RRT converged to a path solution")
                last_child = rrt_tree[child_id]
                break

    current_node = last_child
    while current_node:
        rrt_path.insert(0, current_node.tag)
        current_node = rrt_tree.parent(current_node.identifier)

    return rrt_path, rrt_tree

In the following two code blocks we initialise the robot arm in both the simulator as well the real world. 

The way we do this in the simulator (remember: this is called a plant in traditional control theory) is by getting its data container (formally called a Context) and setting a desired joint value on that data container. 

<div class="alert alert-block alert-info"> <b>Task 11:</b> Run both code cells to verify once more that your real world scenario matches your simulation scenario.</div>

In [ ]:
simulation_data = plant.GetMyContextFromRoot(context)
q_init_new = [0., -np.pi / 2, 0., 0., 0., np.pi / 2]
plant.SetPositions(simulation_data, arm_index, q_init_new)
robot_diagram.ForcedPublish(context)

In [ ]:
robot.move_to_joint_configuration(q_init_new).wait()

<div class="alert alert-block alert-info"> <b>Task 12:</b> Place your microplate somewhere reachable on the table and define a target pose to the microplate. You can define X_UPlate_TCP for this, or define the X_W_TCP_Target directly. Stay 8 cm above the table (or above the microplate) initially to avoid touching the microplate. </div>

Use the visualisation code cell *visualize_frame* to verify the grasp will make sense.

In [ ]:
# target: on table microplate
X_W_TCP_target = np.identity(4)

X_W_UPlate = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                                 plant.GetBodyByName("base_link", plant.GetModelInstanceByName(
                                                     "uplate"))).GetAsMatrix4()
visualize_frame(meshcat, "/scenario/X_W_UPlate", X_W_UPlate, length=0.15, radius=0.005)

In [ ]:
# TODO(@Student): fill in the required transform X_UPlate_TCP. Use the visualized frame X_W_UPlate on your meshcat instance
X_UPlate_Tcp = np.zeros((4,4))
X_W_TCP_target = RigidTransform(X_W_UPlate) @ X_UPlate_Tcp

visualize_frame(meshcat, "/scenario/X_W_TCP_target", X_W_TCP_target, length=0.15, radius=0.005, )
print(f"Look at {meshcat.web_url()} to examine target pose X_W_TCP_target")

We will run RRT in C-space but the target pose is specified in task space. Hence we will use IK to first translate the target pose in task space to the required joint values in C-space. We will use the `RobotKinematics` class we used before for this. The IK of this class is optimization-based (remember the course about planning). This implies the solver can potentially fail to find a solution. In general, this is not a problem because behind the scenes, we will try different solvers and apply tricks to massage the problem formulation. The `RobotKinematics` class will tell you when it fails with a warning and return NaN's as joint solutions. Any intermediate info log telling that it fails, is not an issue.  

Let's examine the target pose in the following cell.

<div class="alert alert-block alert-info"> <b>Task 13:</b> Verify that an IK solution was found. If not, the microplate will be out of reach and your planner will crash.</div>

In [ ]:
meshcat.Delete("rrt_path")  # delete the old red path
X_W_B = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(), plant.GetBodyByName("base"))
X_B_TCP_target = X_W_B.inverse() @ RigidTransform(X_W_TCP_target)
q_goal = robot_kinematics.inverse_kinematics(X_B_TCP_target, q_init_new)
success = True if q_goal is not None else None
if success:
    q_goal = q_goal[0] # take first IK solution, we only output 1 
else:
    q_goal = None 
    print("!! Could not find IK solution !!")
print(f"IK result: {q_goal} with success: {success}")
print(f"Look at {robot_kinematics.meshcat.web_url()} to visualize target pose RRT will try to achieve in C-space.")

<div class="alert alert-block alert-info"> <b>Task 14:</b> Run your planner with the standard parameters. If it doesn't converge within 3000 iterations then your grasping scenerio is too complex for these simplified settings. Change your scenario (rack and/or microplate positioning) so that this converges within 3000 steps.</div>

In [ ]:
p, t = plan_with_rrt(q_init_new, q_goal)

<div class="alert alert-block alert-info"> <b>Task 15:</b> Visualise the optimal path in the simulator and observe. Will you always get the same solution? Is the solution optimal? What could you change in the RRT planner to make it more optimal?</div>

Your answer here

In [ ]:
#draw path in meshcat
meshcat.Delete("rrt_path")
color = [1, 0, 0, 1]
for node_nr in range(len(p) - 1):
    start = X_W_B @ RigidTransform(robot_kinematics.forward_kinematics(p[node_nr]))
    end = X_W_B @ RigidTransform(robot_kinematics.forward_kinematics(p[node_nr + 1]))
    draw_line_between_points(meshcat, f"rrt_path/{node_nr}", start.translation(), end.translation(), 0.05, color)

<div class="alert alert-block alert-info"> <b>Task 16:</b> Now execute your path in the real world. What do you observe? How can you make this path more smooth?</div>

In [ ]:
# Slow method: robot stop at each joint pose
for q_step in p:
    robot.move_to_joint_configuration(q_step).wait()

# Fast method, robot blends a little between joint poses
acceleration = 0.2
speed = 0.2
blend = 0.02
params = [acceleration, speed, blend]  # Feel free to edit these params
path_with_params = [list(q) + params for q in p]
#robot.rtde_control.moveJ(path_with_params)

Your answer here

## Part 4 - RRT parameters

<div class="alert alert-block alert-info"> <b>Task 17:</b> Repeat part 3 but with step delta_q=np.deg2rad(1), delta_q=np.deg2rad(3) and delta_q=np.deg2rad(6). What do you observe if you compare the results? Can you explain your observation?</div>

Note that you can run *plan_with_rrt(scenario, X_W_TCP_target,delta_q=np.deg2rad(6))*

Your answer here

<div class="alert alert-block alert-info"> <b>Task 18:</b> Repeat part 3 but with step tolerance=np.deg2rad(6) and tolerance=np.deg2rad(1). What do you observe if you compare the results? Can you explain your observation?</div>

Note that you can run *plan_with_rrt(scenario, X_W_TCP_target,tolerance=np.deg2rad(6))*

Your answer here

<div class="alert alert-block alert-info"> <b>Task 19:</b> Repeat part 3 but with step p=0.0, p=0.1, p=0.5 and p=0.9. What do you observe if you compare the results? Can you explain your observation? Is a high p value always </div>

Note that you can run *plan_with_rrt(scenario, X_W_TCP_target, p=0.)*

Your answer here

<div class="alert alert-block alert-info"> <b>Task 20:</b> Explore the hyperparameters further and try to find a very good parameter combination. What did you choose?</div>

Your answer here

## Part 5 - Pipetting in the cabin
We will now pipet in the cabin on the middle shelf. 

<div class="alert alert-block alert-info"> <b>Task 21:</b> Put the robot back to a safe home position. You can do this using freedrive and reading out the robot joint states, or by just manually defining the joint states and watching the result on the meshcat simulation </div>

In [ ]:
simulation_data = plant.GetMyContextFromRoot(context)
q_init_new = [0., -np.pi / 2, 0., 0., 0., np.pi / 2]
plant.SetPositions(simulation_data, arm_index, q_init_new)
robot_diagram.ForcedPublish(context)

Move the robot in real life to this position (this might already be the case if you have used freedrive)

In [ ]:
robot.move_to_joint_configuration(q_init_new).wait()

<div class="alert alert-block alert-info"> <b>Task 22:</b> Define a new target TCP pose X_W_TCP_target with the gripper 4 cm above the middle shelf with the pipette finger pointing downwards. </div>

You can reuse homework 3 for this. 

In [ ]:
# ###################################################
# Other target: on middle shelf
meshcat.Delete("/scenario")
meshcat.Delete("rrt")
meshcat.Delete("rrt_path")
X_W_LowerShelfOrigin = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(),
                                                 plant.GetBodyByName("lower_shelf_link", plant.GetModelInstanceByName(
                                                     "wooden_rack"))).GetAsMatrix4()
visualize_frame(meshcat, "/scenario/lower_shelf_origin", X_W_LowerShelfOrigin, length=0.15, radius=0.005)

# Solution that specifies Transform matrix manually: 
# TODO(@Student): Fill in transformation from lowershelf to TCP.
X_LowerShelf_TCP = np.array([
    [1, 0, 0, 0.],
    [0, 1, 0, 0.],
    [0, 0, 1, 0.],
    [0, 0, 0, 1]
])
X_W_TCP_target = X_W_LowerShelfOrigin @ X_LowerShelf_TCP

visualize_frame(meshcat, "/scenario/X_W_TCP_target", X_W_TCP_target, length=0.15, radius=0.005, )

Run IK to verify whether the target pose is reachable

In [ ]:
meshcat.Delete("rrt_path")  # delete the old red path
X_W_B = plant.EvalBodyPoseInWorld(plant.CreateDefaultContext(), plant.GetBodyByName("base"))
X_B_TCP_target = X_W_B.inverse() @ RigidTransform(X_W_TCP_target)
q_goal = robot_kinematics.inverse_kinematics(X_B_TCP_target, q_init_new)
success = True if q_goal is not None else None
if success:
    q_goal = q_goal[0] # take first IK solution, we only output 1 
else:
    q_goal = None 
    print("!! Could not find IK solution !!")
print(f"IK result: {q_goal} with success: {success}")
print(f"Look at {robot_kinematics.meshcat.web_url()} to visualize target pose RRT will try to achieve in C-space.")

<div class="alert alert-block alert-info"> <b>Task 23:</b> Run RRT with your hyperparameters to let it converge. Tune them further if necessary. Are they the same as your optimal set of part 4? </div>

In [ ]:
# TODO(@Student): tune parameters of plan_with_rrt further if necessary 
p, t = plan_with_rrt(q_init_new, q_goal, delta_q=np.deg2rad(2), p=0.10)

Your answer here

In [ ]:
#draw path in meshcat
meshcat.Delete("rrt_path")
color = [1, 0, 0, 1]
for node_nr in range(len(p) - 1):
    start = X_W_B @ RigidTransform(robot_kinematics.forward_kinematics(p[node_nr]))
    end = X_W_B @ RigidTransform(robot_kinematics.forward_kinematics(p[node_nr + 1]))
    draw_line_between_points(meshcat, f"rrt_path/{node_nr}", start.translation(), end.translation(), 0.05, color)

If you dare, you can run the path on the real robot with the following cell (reuse code from task 17)

In [ ]:
# TODO(@Student): reuse your code from task 17
for q_step in p:
    robot.move_to_joint_configuration(q_step).wait()

## The end

Well done. You have now implemented and tested an RRT planner. RRT planners are more than two decades old. However, despite your -- maybe frustrating -- experiences, RRT planners are in use today as well. Therefore all kinds of variants have been implemented and tested such as RRT-Connect, RRT with adaptive step size, RRT with various metrics for measuring the distance between two configurations, etc. In the following practical sessions we will leave this RRT implementation behind and make use of our AIRO motion planning algorithm that uses RRT with nuts and bolts to make it more performant. 